In [201]:
import numpy as np

np.random.seed(0) 

P = [
    [0.9, 0.1, 0.0, 0.0, 0.0, 0.0],
    [0.5, 0.0, 0.5, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.6, 0.0, 0.4],
    [0.0, 0.0, 0.0, 0.0, 0.3, 0.7],
    [0.0, 0.2, 0.3, 0.5, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
]
P = np.array(P) # 转换为numpy数组便于计算
rewards= [-1, -2, -2, 10, 1, 0] # 定义奖励函数
gamma=0.5 # 折扣因子

def compute_return(start_point,chain,gamma):
    G = 0
    for i in reversed(range(start_index, len(chain))):
        G = rewards[chain[i]-1] + gamma * G 
        #这里相当于计算多项式函数求和的递推式，从后向前每一步时间复杂度为O(1)
    return G
chain=[1,2,3,6]
start_index=0
G=compute_return(start_index,chain,gamma)
print("Return G:", G)

Return G: -2.5


In [202]:
def compute_state_value(P,rewards,gamma,states_num): #states_num是状态的数量
    rewards=np.array(rewards).reshape(-1,1) # 将奖励函数转换为列向量
    value = np.dot(np.linalg.inv(np.eye(states_num, states_num) - gamma * P),
                   rewards)
    return value

V=compute_state_value(P,rewards,gamma,len(P))
print("State Values V:\n", V)    

State Values V:
 [[-2.01950168]
 [-2.21451846]
 [ 1.16142785]
 [10.53809283]
 [ 3.58728554]
 [ 0.        ]]


In [203]:
S=['s1','s2','s3','s4','s5']
A=['保持s1','前往s2','前往s3','前往s4','前往s5','概率前往']
P = {#状态转移函数
    "s1-保持s1-s1": 1.0,
    "s1-前往s2-s2": 1.0,
    "s2-前往s1-s1": 1.0,
    "s2-前往s3-s3": 1.0,
    "s3-前往s4-s4": 1.0,
    "s3-前往s5-s5": 1.0,
    "s4-前往s5-s5": 1.0,
    "s4-概率前往-s2": 0.2,
    "s4-概率前往-s3": 0.4,
    "s4-概率前往-s4": 0.4,
}
R = {#奖励函数
    "s1-保持s1": -1,
    "s1-前往s2": 0,
    "s2-前往s1": -1,
    "s2-前往s3": -2,
    "s3-前往s4": -2,
    "s3-前往s5": 0,
    "s4-前往s5": 10,
    "s4-概率前往": 1,
}
gamma=0.5 # 折扣因子
MDP = (S, A, P, R, gamma) # 定义MDP模型

#策略1,随机策略
Pi_1 = {
    "s1-保持s1": 0.5,
    "s1-前往s2": 0.5,
    "s2-前往s1": 0.5,
    "s2-前往s3": 0.5,
    "s3-前往s4": 0.5,
    "s3-前往s5": 0.5,
    "s4-前往s5": 0.5,
    "s4-概率前往": 0.5,
}
# 策略2
Pi_2 = {
    "s1-保持s1": 0.6,
    "s1-前往s2": 0.4,
    "s2-前往s1": 0.3,
    "s2-前往s3": 0.7,
    "s3-前往s4": 0.5,
    "s3-前往s5": 0.5,
    "s4-前往s5": 0.1,
    "s4-概率前往": 0.9,
}


# 把输入的两个字符串通过“-”连接,便于使用上述定义的P、R变量
def join(str1, str2):
    return str1 + '-' + str2

'''我们直接给出转化后的 MRP 的状态转移矩阵和奖励函数'''
P_from_mdp_to_mrp = [
    [0.5, 0.5, 0.0, 0.0, 0.0],
    [0.5, 0.0, 0.5, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.5, 0.5],
    [0.0, 0.1, 0.2, 0.2, 0.5],
    [0.0, 0.0, 0.0, 0.0, 1.0],
]
P_from_mdp_to_mrp = np.array(P_from_mdp_to_mrp) # 转换为numpy数组便于计算
R_from_mdp_to_mrp = [-0.5, -1.5, -1.0, 5.5, 0]
V=compute_state_value(P_from_mdp_to_mrp,R_from_mdp_to_mrp,gamma,len(S))
print("State Values V under Pi_1:\n", V)

State Values V under Pi_1:
 [[-1.22555411]
 [-1.67666232]
 [ 0.51890482]
 [ 6.0756193 ]
 [ 0.        ]]


In [204]:
def sample(MDP,Pi,time_steps_max,numbers): 
    #MDP元组5个参数；Pi策略，上面有两个；time_steps_max每一轮最大轨迹步数；numbers轨迹条数
    S, A, P, R, gamma = MDP
    episodes = []
    for _ in range(numbers):
        episode = []
        timestep=0
        s=S[np.random.randint(4)] # 理论：MDP初始状态分布（此处简化为均匀分布且假设s5为终点）
        while timestep<=time_steps_max and s!='s5':
            timestep+=1
            rand,temp=np.random.rand(),0 #概率过程1，在策略s下按概率选择动作
            for a_opt in A:
                temp+=Pi.get(join(s,a_opt),0) #累计概率法，降低时间复杂度，
                #由于区间长度与概率大小成正比，这样并不改变概率的分布，而且这样也不会选到概率为0的动作
                if temp>rand:
                    a=a_opt
                    r = R.get(join(s, a), 0) #奖励函数
                    break
            rand,temp=np.random.rand(),0 #概率过程2，在状态转移过程按P(s'|s,a)概率转移到下一个状态
            for s_opt in S:
                temp+=P.get(join(join(s,a),s_opt),0) #同样是累计概率法，这样也不会选到状态转移概率为0的next
                if temp>rand:
                    s_next=s_opt
                    break
            episode.append((s,a,r,s_next)) #记录一次状态转移过程
            s=s_next
        episodes.append(episode) #记录一条状态序列（包含多个转移）
    return episodes
episodes = sample(MDP, Pi_1, 20, 5)
print('第一条序列\n', episodes[0])
print('第二条序列\n', episodes[1])
print('第五条序列\n', episodes[4])



第一条序列
 [('s1', '前往s2', 0, 's2'), ('s2', '前往s2', 0, 's2'), ('s2', '前往s2', 0, 's2'), ('s2', '前往s3', -2, 's3'), ('s3', '前往s4', -2, 's4'), ('s4', '概率前往', 1, 's3'), ('s3', '前往s4', -2, 's4'), ('s4', '前往s5', 10, 's5')]
第二条序列
 [('s1', '前往s2', 0, 's2'), ('s2', '前往s2', 0, 's2'), ('s2', '前往s2', 0, 's2'), ('s2', '前往s2', 0, 's2'), ('s2', '前往s2', 0, 's2'), ('s2', '前往s2', 0, 's2'), ('s2', '前往s3', -2, 's3'), ('s3', '前往s5', 0, 's5')]
第五条序列
 [('s3', '前往s5', 0, 's5')]


In [205]:
def MC(episodes,V,N,gamma):
    for episode in episodes: #遍历每条轨迹
        G=0
        for i in range(len(episode)-1,-1,-1): #从后向前，第二章代码也有，O(1)时间计算指数数列求和，计算期望V(s)
            (s,a,r,s_next)=episode[i]
            G=r+gamma*G
            N[s]+=1
            V[s]+=1/N[s]*(G-V[s])

time_steps_max=20

episodes=sample(MDP,Pi_1,time_steps_max,1000)

gamma=0.5
V = {"s1": 0, "s2": 0, "s3": 0, "s4": 0, "s5": 0}
N = {"s1": 0, "s2": 0, "s3": 0, "s4": 0, "s5": 0}
MC(episodes,V,N,gamma)
print("V\n:",V)
print("N\n",N)



V
: {'s1': -1.1007424179388565, 's2': -0.04708992604459269, 's3': 0.5651664426603852, 's4': 6.289949707607735, 's5': 0}
N
 {'s1': 490, 's2': 890, 's3': 869, 's4': 847, 's5': 0}
